In [1]:
"""
price - target(listing price prediction)
In here I only use clean dataset .parquet 
mallorca_listings_clean.parquet
"""

'\nprice - target(listing price prediction)\nIn here I only use clean dataset .parquet \nmallorca_listings_clean.parquet\n'

# Import Libaries

In [2]:
import polars as pl
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_validate

# Analyze basic stat of dataset

In [3]:
df = pl.read_parquet("/home/naviya-c/Desktop/airbnb_fixed/data/interim/mallorca_listings_clean.parquet")
pdf = df.to_pandas()

In [4]:
pdf.shape

(14817, 101)

In [5]:
pdf['neighborhood_overview'].isna().sum()

np.int64(14817)

# Feature Engineer

In [6]:
pdf_1 = pdf.loc[:, pdf.isnull().mean() <= 0.9]

In [7]:
numeric = pdf_1.select_dtypes(include=["number"])

corr = (
    numeric.corr(numeric_only=True)["price"]
    .drop("price")
    .sort_values(key=abs, ascending=False)
)

remove = corr[abs(corr) <= 0.3].index

pdf_2 = pdf_1.drop(remove, axis = 1)

#### Remove threshold < 0.3

In [8]:
pdf_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 14817 entries, 0 to 14816
Data columns (total 58 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   city_key                                      14817 non-null  str           
 1   snapshot_date                                 14817 non-null  str           
 2   listing_url                                   14817 non-null  str           
 3   scrape_id                                     14817 non-null  str           
 4   last_scraped                                  14817 non-null  datetime64[ms]
 5   source                                        14817 non-null  str           
 6   name                                          14817 non-null  str           
 7   description                                   14529 non-null  str           
 8   picture_url                                   14817 non-null  str           


In [9]:
removing_col = pdf_2.columns[:13]

In [10]:
pdf_3 = pdf_2.drop(removing_col, axis = 1)

In [11]:
pdf_3.info()

<class 'pandas.DataFrame'>
RangeIndex: 14817 entries, 0 to 14816
Data columns (total 45 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   hosts_time_as_user_years                      14817 non-null  str           
 1   hosts_time_as_user_months                     14817 non-null  str           
 2   hosts_time_as_host_years                      14817 non-null  str           
 3   hosts_time_as_host_months                     14817 non-null  str           
 4   host_location                                 10902 non-null  str           
 5   host_about                                    10422 non-null  str           
 6   host_is_superhost                             14817 non-null  bool          
 7   host_picture_url                              14817 non-null  str           
 8   host_listings_count                           14817 non-null  int64         


In [12]:
remove_col = ['hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_months','host_is_superhost_missing', 'bathrooms_text', 'has_availability', 'host_location', 'host_has_profile_pic', 'maximum_minimum_nights', 'calendar_last_scraped','beds_filled','bedrooms_filled', 'price_quote_checkin_date', 'price_quote_checkout_date', 'first_review', 'last_review', 'price_quote_raw', 'host_picture_url', 'host_about', 'price_quote_total_price', 'price_quote_price_per_night']

In [13]:
pdf_4 = pdf_3.drop(remove_col, axis=1)

In [14]:
pdf_4.shape

(14817, 24)

In [15]:
pdf_4.info()

<class 'pandas.DataFrame'>
RangeIndex: 14817 entries, 0 to 14816
Data columns (total 24 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   hosts_time_as_host_years                      14817 non-null  str    
 1   host_is_superhost                             14817 non-null  bool   
 2   host_listings_count                           14817 non-null  int64  
 3   host_identity_verified                        14817 non-null  bool   
 4   neighbourhood_cleansed                        14817 non-null  str    
 5   property_type                                 14817 non-null  str    
 6   room_type                                     14817 non-null  str    
 7   accommodates                                  14817 non-null  int64  
 8   bathrooms                                     13155 non-null  str    
 9   bedrooms                                      14320 non-null  float64
 1

In [16]:
pdf_5 = pdf_4.dropna(subset=['price'])

In [17]:
pdf_5.shape

(11057, 24)

In [18]:
pdf_5.info()

<class 'pandas.DataFrame'>
Index: 11057 entries, 1 to 14805
Data columns (total 24 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   hosts_time_as_host_years                      11057 non-null  str    
 1   host_is_superhost                             11057 non-null  bool   
 2   host_listings_count                           11057 non-null  int64  
 3   host_identity_verified                        11057 non-null  bool   
 4   neighbourhood_cleansed                        11057 non-null  str    
 5   property_type                                 11057 non-null  str    
 6   room_type                                     11057 non-null  str    
 7   accommodates                                  11057 non-null  int64  
 8   bathrooms                                     10145 non-null  str    
 9   bedrooms                                      10713 non-null  float64
 10  be

In [19]:
pdf_5.head()

,hosts_time_as_host_years,host_is_superhost,host_listings_count,host_identity_verified,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bedrooms,...,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,host_identity_verified_missing,property_type_clean,property_family,neighbourhood_std,bedrooms_imputed,beds_imputed
1,1,False,1,True,Búger,Entire chalet,Entire home/apt,6,2.5,3.0,...,1,1,0,0,False,Entire Chalet,Entire place,Búger,False,False
2,2,False,219,True,Pollença,Entire villa,Entire home/apt,2,1.0,1.0,...,40,40,0,0,False,Entire Villa,Entire place,Pollença,False,False
3,11,True,24,True,Sant Llorenç des Cardassar,Entire home,Entire home/apt,8,3.0,4.0,...,24,24,0,0,False,Entire Home,Entire place,Sant Llorenç des Cardassar,False,False
4,11,False,57,True,Pollença,Entire home,Entire home/apt,5,NaN,3.0,...,57,57,0,0,False,Entire Home,Entire place,Pollença,False,False
5,10,False,160,True,Santanyí,Entire villa,Entire home/apt,8,4.0,4.0,...,159,159,0,0,False,Entire Villa,Entire place,Santanyí,False,False


In [20]:
pdf_5['bathrooms']

1        2.5
2        1.0
3        3.0
4        NaN
5        4.0
        ... 
14769    NaN
14779    NaN
14780    NaN
14783    NaN
14805    NaN
Name: bathrooms, Length: 11057, dtype: str

In [21]:
pdf_5["bathrooms"] = pd.to_numeric(pdf_5["bathrooms"], errors="coerce")
pdf_5["hosts_time_as_host_years"] = pd.to_numeric(pdf_5["hosts_time_as_host_years"], errors="coerce")

In [22]:
pdf_5.info()

<class 'pandas.DataFrame'>
Index: 11057 entries, 1 to 14805
Data columns (total 24 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   hosts_time_as_host_years                      11057 non-null  int64  
 1   host_is_superhost                             11057 non-null  bool   
 2   host_listings_count                           11057 non-null  int64  
 3   host_identity_verified                        11057 non-null  bool   
 4   neighbourhood_cleansed                        11057 non-null  str    
 5   property_type                                 11057 non-null  str    
 6   room_type                                     11057 non-null  str    
 7   accommodates                                  11057 non-null  int64  
 8   bathrooms                                     10145 non-null  float64
 9   bedrooms                                      10713 non-null  float64
 10  be

In [23]:
pdf_5['license'].unique()

<ArrowStringArray>
[    'Spain - National registration number<br />ESFCTU000007022000236176000000000000000000000ETV/4112<br /><br />Mallorca - Regional registration number<br />ETV/411',
     'Mallorca - Regional registration number<br />ETV/654<br /><br />Spain - National registration number<br />ESFCTU000007030000313469000000000000000000000ETV/6543',
    'Spain - National registration number<br />ESFCTU00000702300017664600000000000000000000ETV/35624<br /><br />Mallorca - Regional registration number<br />ETV/3562',
    'Spain - National registration number<br />ESFCTU00000703000015775900000000000000000000ETV/16960<br /><br />Mallorca - Regional registration number<br />ETV/1696',
    'Spain - National registration number<br />ESFCTU00000700800074055400000000000000000000ETV/80677<br /><br />Mallorca - Regional registration number<br />ETV/8067',
     'Spain - National registration number<br />ESFCTU000007030000046886000000000000000000000ETV/1056<br /><br />Mallorca - Regional registra

In [24]:
pdf_5["license"] = (
    pdf_5["license"]
      .fillna("")
      .str.contains(r"\b(?:ESFC|ETV)\b", case=False)
)

In [25]:
pdf_5.info()

<class 'pandas.DataFrame'>
Index: 11057 entries, 1 to 14805
Data columns (total 24 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   hosts_time_as_host_years                      11057 non-null  int64  
 1   host_is_superhost                             11057 non-null  bool   
 2   host_listings_count                           11057 non-null  int64  
 3   host_identity_verified                        11057 non-null  bool   
 4   neighbourhood_cleansed                        11057 non-null  str    
 5   property_type                                 11057 non-null  str    
 6   room_type                                     11057 non-null  str    
 7   accommodates                                  11057 non-null  int64  
 8   bathrooms                                     10145 non-null  float64
 9   bedrooms                                      10713 non-null  float64
 10  be

In [26]:
pdf_5["bathrooms"] = pdf_5["bathrooms"].fillna(pdf_5["bathrooms"].median())


In [27]:
# Bedrooms
pdf_5["bedrooms_imputed"] = pdf_5["bedrooms"].isna()
pdf_5["bedrooms"] = pdf_5["bedrooms"].fillna(pdf_5["bedrooms"].median())

# Beds
pdf_5["beds_imputed"] = pdf_5["beds"].isna()
pdf_5["beds"] = pdf_5["beds"].fillna(pdf_5["beds"].median())

# Bathrooms
pdf_5["bathrooms_imputed"] = pdf_5["bathrooms"].isna()
pdf_5["bathrooms"] = pdf_5["bathrooms"].fillna(pdf_5["bathrooms"].median())

In [28]:
pdf_5.info()

<class 'pandas.DataFrame'>
Index: 11057 entries, 1 to 14805
Data columns (total 25 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   hosts_time_as_host_years                      11057 non-null  int64  
 1   host_is_superhost                             11057 non-null  bool   
 2   host_listings_count                           11057 non-null  int64  
 3   host_identity_verified                        11057 non-null  bool   
 4   neighbourhood_cleansed                        11057 non-null  str    
 5   property_type                                 11057 non-null  str    
 6   room_type                                     11057 non-null  str    
 7   accommodates                                  11057 non-null  int64  
 8   bathrooms                                     11057 non-null  float64
 9   bedrooms                                      11057 non-null  float64
 10  be

In [29]:
target = pdf_5['price']
features = pdf_5.drop('price', axis = 1)

In [30]:
features['neighbourhood_cleansed'].unique().shape

(53,)

In [31]:
features = pd.get_dummies(
    features,
    columns=[
        "room_type",
        "property_type",
        "neighbourhood_cleansed",
        "property_type_clean",
        "property_family",
        "neighbourhood_std",
    ],
    drop_first=True,
    dtype=bool,
)

In [32]:
features.shape

(11057, 233)

In [33]:
important_amenities = [
    "Wifi",
    "Kitchen",
    "Air conditioning",
    "Heating",
    "TV",
    "Washer",
    "Dryer",
    "Dishwasher",
    "Refrigerator",
    "Microwave",
    "Oven",
    "Coffee maker",
    "Free parking on premises",
    "Pool",
    "Hot tub",
    "Gym",
    "BBQ grill",
    "Outdoor dining area",
    "Outdoor furniture",
    "Patio or balcony",
    "Garden view",
    "Beach access",
    "Waterfront",
    "Dedicated workspace",
    "Indoor fireplace",
    "Elevator",
    "Smoke alarm",
    "Carbon monoxide alarm",
    "First aid kit",
    "Fire extinguisher",
    "Hair dryer",
    "Iron",
    "Hot water",
    "Crib",
    "High chair",
    "Pets allowed",
    "Self check-in",
    "Lockbox",
    "Private entrance",
    "EV charger"
]

In [34]:
import re

for amenity in important_amenities:
    col = "has_" + re.sub(r"[^a-z0-9]+", "_", amenity.lower()).strip("_")

    features[col] = features["amenities"].apply(
        lambda x: any(amenity.lower() in item.lower() for item in x)
    )

features = features.drop(columns=["amenities"])

In [35]:
features.shape

(11057, 272)

# Train Model

In [36]:
target = np.log1p(target)

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
)

In [38]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [39]:
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAPE": "neg_mean_absolute_percentage_error",
}

In [40]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
    ),
}

In [41]:
import pandas as pd

results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )

    results.append({
        "Model": name,
        "MAE": -scores["test_MAE"].mean(),
        "RMSE": -scores["test_RMSE"].mean(),
        "MAPE": -scores["test_MAPE"].mean(),
    })

results = pd.DataFrame(results)
results.sort_values("RMSE")

,Model,MAE,RMSE,MAPE
2,XGBoost,0.368263,0.489352,0.061723
1,Random Forest,0.372167,0.501647,0.062135
0,Linear Regression,0.415317,0.549845,0.069638


In [43]:
best_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)

best_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth